#### Multi-Agentic-Ai System based Blog Generator
<p>
This project is a multi-agent Agentic AI system built using LangGraph, where multiple AI agents collaborate to complete a task.
The system includes a Planner Agent, Research Agent (with RAG tool), and Writer Agent, all connected through a stateful graph workflow.
Each agent performs a specific role and updates a shared state, enabling structured execution from planning to final output generation.
</p>

In [2]:
## First Understand Basics
from typing import TypedDict

class AgentState(TypedDict):
    topic: str
    plan: str
    research: str
    blog: str

In [ ]:
AgentState.__annotations__ 
# to initialize a dictionary, where agents will maintain the state

{'topic': str, 'plan': str, 'research': str, 'blog': str}

In [3]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# ------------------ STATE ------------------
class AgentState(TypedDict):
    topic: str
    plan: str
    research: str
    blog: str


llm = ChatOpenAI(model="gpt-4o-mini")

# ------------------ TOOL (RAG as tool) ------------------
def create_vector_store():
    docs = [
        "AI in healthcare improves diagnosis and treatment.",
        "Machine learning helps in detecting diseases early.",
        "DSA is a very important for software development,",
        "Mathmatics Geometric formula"
      
    ]
    embeddings = OpenAIEmbeddings()
    return FAISS.from_texts(docs, embeddings)

vector_store = create_vector_store()


# ------------------ NODES ------------------

# 1. Planner Agent
def planner(state: AgentState):
    response = llm.invoke(f"Create a step-by-step research plan for: {state['topic']}")
    return {"plan": response.content}


# 2. Research Agent (USES TOOL 🔥)
def researcher(state: AgentState):
    docs = vector_store.similarity_search(state["topic"], k=2)
    context = "\n".join([d.page_content for d in docs])

    response = llm.invoke(f"""
Use this context to research:

{context}

Follow this plan:
{state['plan']}
""")
    return {"research": response.content}


# 3. Writer Agent
def writer(state: AgentState):
    response = llm.invoke(f"""
Write a structured blog using this research:

{state['research']}
""")
    return {"blog": response.content}


# ------------------ GRAPH ------------------

builder = StateGraph(AgentState)

builder.add_node("planner", planner)
builder.add_node("researcher", researcher)
builder.add_node("writer", writer)

builder.set_entry_point("planner")

builder.add_edge("planner", "researcher")
builder.add_edge("researcher", "writer")
builder.add_edge("writer", END)

graph = builder.compile()

# ------------------ RUN ------------------

result = graph.invoke({"topic": "AI in healthcare"})

print("\n--- FINAL BLOG ---\n")
print(result["blog"])

: 

### Thank You